# MicrobeAtlas Metal–Microbial Community Analysis Supplement
## Sensitivity Analysis
Loads pre‑computed matrices from disk. No Spark required.

#### Tests:
- single‑metal
- multiple imputation
- complete‑case

In [3]:
import os, json, logging, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from scipy.stats import rankdata, combine_pvalues
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
warnings.filterwarnings('ignore')

log = logging.getLogger()
logging.basicConfig(level=logging.INFO)

# Paths to saved intermediate files
PROJ_ROOT = Path.cwd().parent
DATA_DIR = PROJ_ROOT / 'data'
FIG_DIR  = PROJ_ROOT / 'figures'
sensitivity_dir = DATA_DIR / 'sensitivity_inputs'

output_dir = DATA_DIR / "sensitivity_output"
os.makedirs(output_dir, exist_ok=True)

clr_path = f"{sensitivity_dir}/clr_test.parquet"            
top_otus_path = f"{sensitivity_dir}/top_otus.csv"
common_idx_path = f"{sensitivity_dir}/common_idx.csv"
geo_metals_raw_path = f"{sensitivity_dir}/geo_metals_pd.parquet"
geo_metals_scaled_path = f"{sensitivity_dir}/geo_metals_scaled.parquet"
cov_scaled_path = f"{sensitivity_dir}/cov_matrix_scaled.parquet"


threshold_file = sensitivity_dir / "null_thresholds.json"
if threshold_file.exists():
    with open(threshold_file) as f:
        saved = json.load(f)
    effect_threshold = saved["threshold_95"]
    log.info(f"Loaded data‑driven threshold from main analysis: {effect_threshold:.4f}")
else:
    effect_threshold = 0.1   # conservative fallback
    log.warn("Warning: null_thresholds.json not found. Using default threshold 0.1.")
    
# Configuration 
CONFIG = {
    'METALS': ["Co", "Cr", "Cu", "Ni", "Zn", "Pb", "As", "Cd", "Hg"],
    'FDR_ALPHA': 0.05,
    'EFFECT_SIZE_THRESHOLD': effect_threshold, # from null distribution in main
    'N_PERMUTATION_SENS': 1000,                # fewer permutations for sensitivity
    'N_IMPUTATIONS': 5,
    'SEED': 42,
    'PCA_COMPONENTS': 20,
}


log.info("Loading pre‑computed data …")
clr_test = pd.read_parquet(clr_path)
top_otus = pd.read_csv(top_otus_path)['otu_id'].tolist()
common_idx = pd.read_csv(common_idx_path)['sample_id'].tolist()
geo_metals_pd = pd.read_parquet(geo_metals_raw_path)
geo_metals_scaled = pd.read_parquet(geo_metals_scaled_path)
cov_matrix_scaled = pd.read_parquet(cov_scaled_path)

# Align indices (ensure all are consistent with common_idx)
common_idx = list(set(common_idx) & set(geo_metals_pd.index) & set(cov_matrix_scaled.index) & set(clr_test.index))
clr_test = clr_test.loc[common_idx]
geo_metals_pd = geo_metals_pd.loc[common_idx]
geo_metals_scaled = geo_metals_scaled.loc[common_idx]
cov_matrix_scaled = cov_matrix_scaled.loc[common_idx]

log.info(f"Samples after alignment: {len(common_idx)}")
log.info(f"Top OTUs: {len(top_otus)}")
log.info(f"Metals: {CONFIG['METALS']}")

Loading pre‑computed data …


Samples after alignment: 71199
Top OTUs: 2000
Metals: ['Co', 'Cr', 'Cu', 'Ni', 'Zn', 'Pb', 'As', 'Cd', 'Hg']


In [4]:
# ─── Helper Functions ─────────────────────────────────────────────────────────

def rankdata_array(x):
    """Column‑wise ranking."""
    return np.apply_along_axis(rankdata, 0, x)


def residualise_ranks(Y, Z_rank):
    Z_design = np.column_stack([np.ones(Z_rank.shape[0]), Z_rank])
    beta, _, rank, _ = np.linalg.lstsq(Z_design, Y, rcond=None)
    Y_pred = Z_design @ beta
    return Y - Y_pred


def partial_spearman_matrix_resid(x, Y, Z, n_perm=9999, seed=None, pca_components=None):
    """
    Partial Spearman correlations between x (1D) and each column of Y,
    controlling for Z. Applies PCA to Z if pca_components is given.
    Returns r_obs (array) and p_perm (array of permutation p‑values).
    """
    rng = np.random.default_rng(seed)
    n = len(x)

    if pca_components is not None:
        scaler = StandardScaler()
        Z_scaled = scaler.fit_transform(Z)
        max_comp = min(pca_components, Z.shape[1], n-2)
        if max_comp > 0:
            pca = PCA(n_components=max_comp)
            Z = pca.fit_transform(Z_scaled)
        else:
            Z = Z_scaled

    Z_rank = rankdata_array(Z)
    Z_design = np.column_stack([np.ones(n), Z_rank])

    Y_res = residualise_ranks(Y, Z_rank)

    x_rank = rankdata(x)
    beta_x, _, _, _ = np.linalg.lstsq(Z_design, x_rank, rcond=None)
    x_res = x_rank - Z_design @ beta_x

    x_std = (x_res - x_res.mean()) / (x_res.std(ddof=1) + 1e-300)
    Y_std = (Y_res - Y_res.mean(axis=0)) / (Y_res.std(axis=0, ddof=1) + 1e-300)
    r_obs = (x_std @ Y_std) / (n - 1)

    if n_perm == 0:
        return r_obs, np.full(Y.shape[1], np.nan)

    perm_stats = np.empty((n_perm, Y.shape[1]))
    for i in range(n_perm):
        perm_x = rng.permutation(x_std)
        perm_stats[i] = perm_x @ Y_std / (n - 1)

    p_perm = (np.abs(perm_stats) >= np.abs(r_obs)).mean(axis=0)
    return r_obs, p_perm


def apply_fdr_and_filter(res_df, effect_threshold, fdr_alpha=0.05):
    """Per‑metal BH FDR and effect‑size threshold."""
    fdr_adj = []
    for metal, sub in res_df.groupby("exposure"):
        _, p_adj, _, _ = multipletests(sub["p_value_perm"], method="fdr_bh")
        fdr_adj.append(pd.Series(p_adj, index=sub.index, name="p_adj"))
    res_df["p_adj"] = pd.concat(fdr_adj).sort_index()
    sig = res_df[(res_df["p_adj"] < fdr_alpha) &
                 (res_df["partial_r"].abs() >= effect_threshold)]
    return res_df, sig

In [5]:
# ─── Complete-case analysis ───────────────────────────────────────────────────

def run_complete_case():
    log.info("Running complete‑case analysis (mutual adjustment) …")
    complete_mask = geo_metals_pd[CONFIG['METALS']].notna().all(axis=1)
    idx = geo_metals_pd.index[complete_mask]
    n_valid = len(idx)
    if n_valid < 20:
        log.error("Too few complete samples.")
        return None, None

    all_results = []
    rng = np.random.default_rng(CONFIG['SEED'])

    for metal in tqdm(CONFIG['METALS'], desc="Metals (complete)"):
        x = geo_metals_pd.loc[idx, metal].values
        other_metals = [m for m in CONFIG['METALS'] if m != metal]
        Z_other = geo_metals_scaled.loc[idx, other_metals].values
        Z_cov = cov_matrix_scaled.loc[idx].values
        Z = np.column_stack([Z_cov, Z_other])

        # PCA reduction inside the function
        Y = clr_test.loc[idx].values

        r_obs, p_perm = partial_spearman_matrix_resid(
            x, Y, Z,
            n_perm=CONFIG['N_PERMUTATION_SENS'],
            seed=int(rng.integers(1_000_000)),
            pca_components=CONFIG['PCA_COMPONENTS']
        )
        for i, otu in enumerate(top_otus):
            all_results.append({
                'otu_id': otu,
                'exposure': metal,
                'partial_r': r_obs[i],
                'p_value_perm': p_perm[i],
                'n': n_valid,
            })

    res_df = pd.DataFrame(all_results)
    res_df, sig = apply_fdr_and_filter(res_df, CONFIG['EFFECT_SIZE_THRESHOLD'], CONFIG['FDR_ALPHA'])
    log.info(f"Complete‑case significant: {len(sig)}")
    return res_df, sig

res_cc, sig_cc = run_complete_case()

INFO:root:Running complete‑case analysis (mutual adjustment) …


Metals (complete):   0%|          | 0/9 [00:00<?, ?it/s]

Metals (complete):  11%|█         | 1/9 [00:30<04:04, 30.56s/it]

Metals (complete):  22%|██▏       | 2/9 [00:45<02:28, 21.17s/it]

Metals (complete):  33%|███▎      | 3/9 [00:58<01:46, 17.77s/it]

...

Metals (complete):  78%|███████▊  | 7/9 [01:55<00:29, 14.71s/it]

Metals (complete):  89%|████████▉ | 8/9 [02:09<00:14, 14.48s/it]

Metals (complete): 100%|██████████| 9/9 [02:22<00:00, 14.14s/it]

Metals (complete): 100%|██████████| 9/9 [02:22<00:00, 15.86s/it]


INFO:root:Complete‑case significant: 1374


In [6]:
# ─── Single-metal models ──────────────────────────────────────────────────────

def run_single_metal(cache_path=output_dir / "cache_single_metal.parquet"):
    """Single‑metal analysis with incremental caching."""
    log.info("Running single‑metal models (cached) …")
    
    # Load existing cache
    if os.path.exists(cache_path):
        existing = pd.read_parquet(cache_path)
        metals_done = set(existing["exposure"].unique())
        log.info(f"Loaded cache: metals already done: {metals_done}")
    else:
        existing = pd.DataFrame(columns=["otu_id","exposure","partial_r","p_value_perm","n"])
        metals_done = set()
    
    metals_to_run = [m for m in CONFIG['METALS'] if m not in metals_done]
    log.info(f"Metals to compute: {metals_to_run}")
    
    rng = np.random.default_rng(CONFIG['SEED'])
    all_results = existing.copy()
    
    for metal in tqdm(metals_to_run, desc="Metals (single)"):
        metal_mask = geo_metals_pd[metal].notna().values
        idx = geo_metals_pd.index[metal_mask]
        n_valid = len(idx)
        if n_valid < 20:
            log.warning(f"Skipping {metal}: only {n_valid} samples.")
            continue
        
        x = geo_metals_pd.loc[idx, metal].values
        Z_cov = cov_matrix_scaled.loc[idx].values
        Y = clr_test.loc[idx].values
        
        r_obs, p_perm = partial_spearman_matrix_resid(
            x, Y, Z_cov,
            n_perm=CONFIG['N_PERMUTATION_SENS'],
            seed=int(rng.integers(1_000_000)),
            pca_components=CONFIG['PCA_COMPONENTS']
        )
        # Build DataFrame for this metal
        metal_df = pd.DataFrame({
            'otu_id': top_otus,
            'exposure': metal,
            'partial_r': r_obs,
            'p_value_perm': p_perm,
            'n': n_valid,
        })
        # Append to existing results
        all_results = pd.concat([all_results, metal_df], ignore_index=True)
        # Save cache immediately
        all_results.to_parquet(cache_path)
        log.info(f"Saved {len(metal_df)} results for {metal}. Total results: {len(all_results)}")
    
    res_df = all_results.copy()
    res_df, sig = apply_fdr_and_filter(res_df, CONFIG['EFFECT_SIZE_THRESHOLD'], CONFIG['FDR_ALPHA'])
    log.info(f"Single‑metal significant: {len(sig)}")
    return res_df, sig

res_single, sig_single = run_single_metal(cache_path=output_dir / "cache_single_metal.parquet")

INFO:root:Running single‑metal models …


Metals (single):   0%|          | 0/9 [00:00<?, ?it/s]

Metals (single):  11%|█         | 1/9 [06:31<52:12, 391.60s/it]

Metals (single):  22%|██▏       | 2/9 [13:52<49:03, 420.54s/it]

Metals (single):  33%|███▎      | 3/9 [20:02<39:45, 397.60s/it]

...

Metals (single):  78%|███████▊  | 7/9 [43:27<10:42, 321.02s/it]

Metals (single):  89%|████████▉ | 8/9 [45:26<04:16, 256.68s/it]

Metals (single): 100%|██████████| 9/9 [45:42<00:00, 181.44s/it]

Metals (single): 100%|██████████| 9/9 [45:42<00:00, 304.77s/it]


INFO:root:Single‑metal significant: 369


In [7]:
# ─── Multiple imputation ──────────────────────────────────────────────────────

def run_imputed(n_imputations=5, cache_path=output_dir / "cache_imputed.parquet"):
    """Multiple imputation with incremental caching per imputation & metal."""
    log.info(f"Running multiple imputation ({n_imputations} imputations, cached) …")
    
    # Load existing cache
    if os.path.exists(cache_path):
        existing = pd.read_parquet(cache_path)
        done_pairs = set(existing[['imputation','exposure']].drop_duplicates().apply(tuple, axis=1))
        log.info(f"Loaded cache: already completed (imp, metal) pairs: {len(done_pairs)}")
    else:
        existing = pd.DataFrame(columns=["otu_id","exposure","partial_r","p_value_perm","n","imputation"])
        done_pairs = set()
    
    metal_data = geo_metals_pd[CONFIG['METALS']].copy()
    cov_data = cov_matrix_scaled.loc[metal_data.index].copy()
    impute_data = pd.concat([metal_data, cov_data], axis=1)
    
    all_results = existing.copy()
    
    for imp in range(n_imputations):
        # Determine which metals still need processing for this imputation
        metals_to_do = [m for m in CONFIG['METALS'] if (imp, m) not in done_pairs]
        if not metals_to_do:
            log.info(f"Imputation {imp+1}: all metals already cached.")
            continue
        
        log.info(f"Imputation {imp+1}/{n_imputations}: processing {metals_to_do}")
        # Different seed for each imputation
        imputer = IterativeImputer(max_iter=20, random_state=CONFIG['SEED'] + imp)
        imputed_array = imputer.fit_transform(impute_data)
        imputed_metals = pd.DataFrame(
            imputed_array[:, :len(CONFIG['METALS'])],
            index=metal_data.index, columns=CONFIG['METALS']
        ).clip(lower=0)
        
        idx = imputed_metals.index
        rng = np.random.default_rng(CONFIG['SEED'] + imp)
        
        for metal in tqdm(metals_to_do, desc=f"Imp {imp+1}"):
            x = imputed_metals.loc[idx, metal].values
            other_metals = [m for m in CONFIG['METALS'] if m != metal]
            Z_other = imputed_metals.loc[idx, other_metals].values
            Z_cov = cov_matrix_scaled.loc[idx].values
            Z = np.column_stack([Z_cov, Z_other])
            Y = clr_test.loc[idx].values
            
            r_obs, p_perm = partial_spearman_matrix_resid(
                x, Y, Z,
                n_perm=CONFIG['N_PERMUTATION_SENS'],
                seed=int(rng.integers(1_000_000)),
                pca_components=CONFIG['PCA_COMPONENTS']
            )
            metal_df = pd.DataFrame({
                'otu_id': top_otus,
                'exposure': metal,
                'partial_r': r_obs,
                'p_value_perm': p_perm,
                'n': len(idx),
                'imputation': imp,
            })
            all_results = pd.concat([all_results, metal_df], ignore_index=True)
            # Save after each metal – robust to crashes
            all_results.to_parquet(cache_path)
            log.info(f"  Saved {metal} (imp {imp+1})")
    
    # Pooling (uses all results collected so far)
    pooled_list = []
    for (otu, metal), group in all_results.groupby(['otu_id', 'exposure']):
        z = np.arctanh(group['partial_r'].clip(-0.999, 0.999))
        mean_z = np.mean(z)
        pooled_r = np.tanh(mean_z)
        chi2, p_comb = combine_pvalues(group['p_value_perm'], method='fisher')
        pooled_list.append({
            'otu_id': otu,
            'exposure': metal,
            'partial_r': pooled_r,
            'p_value_perm': p_comb,
            'n': group['n'].iloc[0],
        })
    
    pooled = pd.DataFrame(pooled_list)
    pooled, sig = apply_fdr_and_filter(pooled, CONFIG['EFFECT_SIZE_THRESHOLD'], CONFIG['FDR_ALPHA'])
    log.info(f"Imputed significant: {len(sig)}")
    return pooled, sig

res_imp, sig_imp = run_imputed(n_imputations=CONFIG['N_IMPUTATIONS'],
                               cache_path=output_dir / "cache_imputed.parquet")

INFO:root:Running multiple imputation (5 imputations) …


INFO:root:Imputation 1/5


Imp 1:   0%|          | 0/9 [00:00<?, ?it/s]

Imp 1:  11%|█         | 1/9 [08:23<1:07:04, 503.09s/it]

Imp 1:  22%|██▏       | 2/9 [16:27<57:25, 492.21s/it]  

...

Imp 5:  78%|███████▊  | 7/9 [53:54<15:21, 460.91s/it]

Imp 5:  89%|████████▉ | 8/9 [1:01:32<07:39, 459.92s/it]

Imp 5: 100%|██████████| 9/9 [1:09:12<00:00, 459.91s/it]

Imp 5: 100%|██████████| 9/9 [1:09:12<00:00, 461.38s/it]

INFO:root:Imputed significant: 2


In [8]:
# ─── Compare results and save outputs ─────────────────────────────────────────

def compare_and_save(sig_cc, sig_single, sig_imp, res_cc, res_single, res_imp):
    log = logging.getLogger()
    log.info("\n=== COMPARISON SUMMARY ===")
    log.info(f"Complete‑case significant: {len(sig_cc)}")
    log.info(f"Single‑metal significant: {len(sig_single)}")
    log.info(f"Imputed significant: {len(sig_imp)}")

    cc_keys = set(sig_cc['otu_id'] + '_' + sig_cc['exposure'])
    single_keys = set(sig_single['otu_id'] + '_' + sig_single['exposure'])
    imp_keys = set(sig_imp['otu_id'] + '_' + sig_imp['exposure'])

    overlap_cc_single = len(cc_keys & single_keys)
    overlap_cc_imp = len(cc_keys & imp_keys)
    overlap_all = len(cc_keys & single_keys & imp_keys)

    log.info(f"Overlap CC vs Single: {overlap_cc_single}")
    log.info(f"Overlap CC vs Imputed: {overlap_cc_imp}")
    log.info(f"Overlap all three: {overlap_all}")

    # Correlation of effect sizes
    merged1 = pd.merge(res_cc[['otu_id','exposure','partial_r']],
                       res_single[['otu_id','exposure','partial_r']],
                       on=['otu_id','exposure'], suffixes=('_cc','_single'), how='inner')
    if len(merged1) > 1:
        corr1 = merged1['partial_r_cc'].corr(merged1['partial_r_single'])
        log.info(f"Correlation CC vs Single: {corr1:.3f}")

    merged2 = pd.merge(res_cc[['otu_id','exposure','partial_r']],
                       res_imp[['otu_id','exposure','partial_r']],
                       on=['otu_id','exposure'], suffixes=('_cc','_imp'), how='inner')
    if len(merged2) > 1:
        corr2 = merged2['partial_r_cc'].corr(merged2['partial_r_imp'])
        log.info(f"Correlation CC vs Imputed: {corr2:.3f}")

    # Save results
    sig_cc.to_csv(output_dir / "sig_complete_case.csv", index=False)
    sig_single.to_csv(output_dir / "sig_single_metal.csv", index=False)
    sig_imp.to_csv(output_dir / "sig_imputed.csv", index=False)
    res_cc.to_csv(output_dir / "res_complete_case.csv", index=False)
    res_single.to_csv(output_dir / "res_single_metal_full.csv", index=False)
    res_imp.to_csv(output_dir / "res_imputed_full.csv", index=False)
    log.info(f"Results saved to {output_dir}")

compare_and_save(sig_cc, sig_single, sig_imp, res_cc, res_single, res_imp)

INFO:root:
=== COMPARISON SUMMARY ===


INFO:root:Complete‑case significant: 1374


INFO:root:Single‑metal significant: 369


INFO:root:Imputed significant: 2


INFO:root:Overlap CC vs Single: 51


INFO:root:Overlap CC vs Imputed: 0


INFO:root:Overlap all three: 0


INFO:root:Correlation CC vs Single: 0.107


INFO:root:Correlation CC vs Imputed: 0.012


INFO:root:Results saved to sensitivity_output/
